# reproduce.ipynb —— 从仓库权重到官方提交 CSV 的全流程复现

本 notebook 是复现本仓库提交结果的唯一入口：加载 `weights/` 里的 4 个集成成员
checkpoint，对官方 **test 180 张** 图做 1024² 推理 → 4 seed 概率平均 → 闭运算 →
阈值 → 面积下限 → 4 邻接连通域 →（逐级放宽兜底）→ COCO 压缩 RLE → 写出并校验
官方格式提交 CSV。**除本仓库与比赛数据本身外，不需要任何额外文件。**

## 运行前提
1. 依赖：`pip install -r requirements.txt`（锁版本），然后

       jupyter nbconvert --to notebook --execute reproduce.ipynb --output reproduce_executed.ipynb

   全流程在单卡 V100S-32GB 约 10–15 分钟；纯 CPU 也能跑通（更慢，且推理走 fp32，
   与 GPU fp16 相比可能有毫级的数值差异）。
2. 数据：官方条款禁止数据再分发，请自行从 Kaggle 比赛页下载 `MAGFiLO_1.0_Kaggle_2026`
   （test/test_images 180 张 + train/train_images + train 标注 JSON），然后二选一：
   - 设环境变量 `SOLAR_DATA_ROOT` 指向数据根目录（其下应有
     `MAGFiLO_1.0_Kaggle_2026/test/test_images` 等）；或
   - 把数据放到本仓库的 `./data/`（默认查找位置）。
3. 输出位置：概率缓存与提交 CSV 写到 `$SOLAR_WORK_DIR`（缺省为系统临时目录下的
   `solar_repro_work/`），**不写进仓库**——仓库卫生闸 `check_repo.py` 禁止
   npz/csv 入库。
4. 最后一个单元（可选的 val-88 官方口径 PQ 复现）需要冻结的 val 划分
   （环境变量 `SOLAR_VAL_SPLIT` 指向 `val_split.json`）；没有就自动跳过，
   不影响提交 CSV 的生成。划分也可用仓库 `make_val_split.py` 自行生成。

## 复现口径（与当年落盘提交件一致的关键设定）
- 模型：`FilamentUNet5(base=32, depth=5)`（7,759,653 参数），base/depth 从
  checkpoint 的 `args` 读取，不手工对齐（`unet_train_crop.py` / `unet_infer1024.py`）；
- 推理：2048² jpeg → L → BILINEAR → 1024² → /255，TTA = **hvflip**：原图 + 水平
  翻转 + 垂直翻转 + **（水平∘垂直）对角变体共 4 次前向等权平均**（⚠ 含对角
  变体、不是 3 次——当年产出提交件的原始推理脚本即此口径；本仓库
  `unet_infer1024.py` 已与之对齐，实测对冻结概率缓存逐位复现 |Δ|=0）；
- 集成：s12 → s5 → s11 → s3 的逐像素概率平均，**加法顺序也与当年一致**；
- 后处理：`postproc_sweep.py` 的 `close7_t070_a600` preset = close k=7 →
  θ=0.70 → 最小面积 600 → 4 邻接连通域 → 逐级放宽兜底（保证每图 ≥1 实例）；
- RLE：2048² 定尺 COCO 压缩 RLE、无引号（`rle.py` 纯 python 实现，与
  pycocotools 交叉对拍过）。

## 预期结果
- 提交 CSV：180/180 图覆盖、**1096 数据行**、硬闸 issues=0；
- val-88（可选）：官方池化口径 PQ ≈ **0.3449**（TP 779 / FP 632 / FN 735，
  预测实例 554）。


In [ ]:
import os, sys, json, time, tempfile
import numpy as np

def _repo_root() -> str:
    """本 notebook 位于仓库根目录；从仓库根启动 jupyter / nbconvert 即可。"""
    try:
        return os.path.dirname(os.path.abspath(__file__))
    except NameError:
        cwd = os.path.abspath(os.getcwd())
        if not os.path.exists(os.path.join(cwd, "unet_infer1024.py")):
            raise RuntimeError("请在仓库根目录启动本 notebook（未找到 unet_infer1024.py）")
        return cwd

REPO = _repo_root()
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# 数据根：环境变量 SOLAR_DATA_ROOT 优先，缺省 ./data（README §数据：自行从 Kaggle 下载）
DATA_ROOT = os.environ.get("SOLAR_DATA_ROOT", os.path.join(REPO, "data"))
TEST_IMAGES = os.path.join(DATA_ROOT, "MAGFiLO_1.0_Kaggle_2026", "test", "test_images")
TRAIN_IMAGES = os.path.join(DATA_ROOT, "MAGFiLO_1.0_Kaggle_2026", "train", "train_images")
TRAIN_ANN = os.path.join(DATA_ROOT, "MAGFiLO_1.0_Kaggle_2026", "train",
                         "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
# 工作目录：概率缓存 / 提交 CSV 落这里，绝不写进仓库
WORK = os.environ.get("SOLAR_WORK_DIR") or os.path.join(tempfile.gettempdir(),
                                                        "solar_repro_work")
os.makedirs(WORK, exist_ok=True)

for name, p in [("TEST_IMAGES", TEST_IMAGES), ("TRAIN_IMAGES", TRAIN_IMAGES)]:
    if not os.path.isdir(p):
        raise FileNotFoundError(
            f"{name} 不存在：{p}\n"
            "请从 Kaggle 下载 MAGFiLO_1.0_Kaggle_2026，并设 SOLAR_DATA_ROOT（见顶部说明）")

def _count_images(d: str) -> int:
    return sum(1 for f in os.listdir(d)
               if f.lower().endswith((".jpeg", ".jpg", ".png")))

import torch
import scipy
import PIL

print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "（CPU 推理）")
print("numpy       :", np.__version__)
print("scipy       :", scipy.__version__)
print("pillow      :", PIL.__version__)
print("DATA_ROOT   :", DATA_ROOT)
print("WORK        :", WORK)
print("test  images:", _count_images(TEST_IMAGES), "张（官方固定 180）")
print("train images:", _count_images(TRAIN_IMAGES), "张")

SEEDS = [3, 5, 11, 12]        # 4 个集成成员 = weights/unet_s{S}_best.pt
ENS_ORDER = [12, 5, 11, 3]    # 概率平均顺序（与当年落盘提交一致）
TTA = "hvflip"                # 见顶部「复现口径」
torch.manual_seed(0)
np.random.seed(0)             # 推理本身无随机性，设了只是保险

## 1. 加载 4 个集成成员权重

`unet_infer1024.load_model1024` 直接复用仓库实现：从 checkpoint 的 `args` 读
`base` / `depth` 构造 `FilamentUNet5`，再 `load_state_dict`。4 份权重的
`base=32, depth=5`，各 7,759,653 参数。

In [ ]:
from unet_infer1024 import load_model1024, prob_for_image1024

MODELS = {}
for s in SEEDS:
    ckpt_path = os.path.join(REPO, "weights", f"unet_s{s}_best.pt")
    assert os.path.exists(ckpt_path), f"缺权重 {ckpt_path}"
    model, ck, dev = load_model1024(ckpt_path)          # base/depth 读自 ckpt["args"]
    MODELS[s] = {"model": model, "dev": dev, "ck": ck}
    a = ck.get("args", {}) or {}
    n_par = sum(p.numel() for p in model.parameters())
    print(f"unet_s{s}_best.pt → epoch={ck.get('epoch')} base={a.get('base')} "
          f"depth={a.get('depth')} params={n_par:,} device={dev}")

## 2. test 180 张 × 4 seed 推理 → 概率缓存

逐图输出 `<stem>.npz`（`prob1024` float16 / `cls1024` uint8，1024²），格式与
`unet_infer1024.py` 完全一致；缓存已存在的图自动跳过（可断点续跑）。
GPU 上这一步约 4–6 分钟，纯 CPU 约 5–6 小时（本机实测：单图 1024² 推理约 25–30 s，4 seed × 180 张全程约 5–6 小时）。

In [ ]:
test_paths = {os.path.splitext(f)[0]: os.path.join(TEST_IMAGES, f)
              for f in os.listdir(TEST_IMAGES)
              if f.lower().endswith((".jpeg", ".jpg", ".png"))}
test_ids = sorted(test_paths)
assert len(test_ids) == 180, f"test 图应为 180 张，实际 {len(test_ids)}"

prob_dirs = {}
t_all = time.perf_counter()
for s in SEEDS:
    d = os.path.join(WORK, f"prob_test_s{s}_tta")
    os.makedirs(d, exist_ok=True)
    prob_dirs[s] = d
    t0, fresh = time.perf_counter(), 0
    for i, stem in enumerate(test_ids, 1):
        dst = os.path.join(d, stem + ".npz")
        if os.path.exists(dst):
            continue
        prob, cls = prob_for_image1024(test_paths[stem], MODELS[s]["model"],
                                       MODELS[s]["dev"], tta=TTA)
        np.savez_compressed(dst, prob1024=prob.astype(np.float16),
                            cls1024=cls.astype(np.uint8))
        fresh += 1
        if i % 36 == 0 or i == len(test_ids):
            print(f"[s{s}] {i}/{len(test_ids)}（新算 {fresh}，"
                  f"{time.perf_counter()-t0:.0f}s）", flush=True)
    with open(os.path.join(d, "_meta.json"), "w", encoding="utf-8") as f:
        json.dump({"ckpt_epoch": MODELS[s]["ck"].get("epoch"), "tta": TTA,
                   "n_images": len(test_ids), "device": str(MODELS[s]["dev"]),
                   "crop": 1024}, f, ensure_ascii=False, indent=1)
print(f"4 个 seed × {len(test_ids)} 张概率缓存就绪，总用时 {time.perf_counter()-t_all:.0f}s")

## 3. 4 seed 概率平均 → close7 → θ0.70 → 面积 ≥600 → 实例 → 提交 CSV

直接复用 `postproc_sweep.py` 的实现（概率平均 `avg_sources`、2048² 放大
`upsample2048`、实例抽取 `extract_spec`——含逐级放宽兜底），提交写入走
`unet_infer.write_submission`（内部先过 `submit_writer.validate_submission_csv`
硬闸，不通过会直接抛错）。平均顺序 = s12, s5, s11, s3。

In [ ]:
import postproc_sweep as ps
from unet_infer import labels_to_masks, write_submission

spec = dict(ps.PRESETS["close7_t070_a600"])
print("spec =", spec)

prob_srcs, cls, sources = ps.load_prob_dirs([prob_dirs[s] for s in ENS_ORDER])
missing = [i for i in test_ids if i not in cls]
assert not missing, f"缺概率缓存 {len(missing)} 张：{missing[:5]}"

preds, n_inst, fallback_imgs = {}, 0, 0
t0 = time.perf_counter()
for i, image_id in enumerate(test_ids, 1):
    pr, cl = ps.upsample2048(ps.avg_sources(prob_srcs, None, image_id), cls[image_id])
    lab, n, tag = ps.extract_spec(pr, cl, spec)
    fallback_imgs += int(tag != "primary")
    masks = labels_to_masks(lab)
    preds[image_id] = masks
    n_inst += len(masks)
    if i % 45 == 0 or i == len(test_ids):
        print(f"[postproc] {i}/{len(test_ids)} 实例累计 {n_inst} "
              f"({time.perf_counter()-t0:.0f}s)", flush=True)

OUT_CSV = os.path.join(WORK, "submission_ens4_close7_t070_a600_full180.csv")
info = write_submission(preds, OUT_CSV)
print(f"[submit] {OUT_CSV}")
print(f"图 {info['images']} | 数据行 {info['rows']} | 实例 {n_inst} | "
      f"触发兜底的图 {fallback_imgs} | 硬闸 issues={info['issues']} | "
      f"用时 {time.perf_counter()-t0:.0f}s")

## 4. 提交格式校验 + 覆盖统计 + 样例

用 `submit_writer.validate_submission_csv` 独立再校验一遍落盘的 CSV：
表头/两列/filament_id 唯一/无引号/RLE 可解码且恰为 2048²/掩膜非空，
并核对 180/180 图覆盖。

In [ ]:
from submit_writer import validate_submission_csv

text = open(OUT_CSV, encoding="utf-8").read()
issues = validate_submission_csv(text, expect_images=set(test_ids))
data_lines = text.splitlines()[1:]
covered = {ln.split(",", 1)[0].rsplit("_", 1)[0] for ln in data_lines}

print("validate_submission_csv issues:", issues if issues else "无（硬闸通过）")
print(f"数据行数：{len(data_lines)}（当年落盘提交件 = 1096）")
print(f"覆盖图像：{len(covered)}/180")
print(f"实例总数：{sum(len(v) for v in preds.values())}")
assert not issues, f"硬闸未通过：{issues[:5]}"
assert len(covered) == 180, "覆盖不足 180 图"

print("\n前 3 行（RLE 截断显示）：")
for ln in data_lines[:3]:
    print(" ", ln[:88] + ("…" if len(ln) > 88 else ""))

## 5.（可选）val-88 官方口径 PQ 复现

对冻结的 val-88 名单重复同一配方（4 seed 概率平均 + close7/θ0.70/ma600），
用 `solar_eval_local.py` 的官方口径评测（行 = 标注者×图像、IoU>0.5、TP 数
命中对、分子累加命中对 IoU、全行池化）。当年冻结概率缓存口径：
**PQ = 0.344895，TP 779 / FP 632 / FN 735，预测实例 554**。
需要：训练标注 JSON + `SOLAR_VAL_SPLIT` 指向冻结 `val_split.json`；
缺任一则跳过（不影响提交 CSV）。

In [ ]:
VAL_SPLIT = os.environ.get("SOLAR_VAL_SPLIT") or os.path.join(WORK, "labels",
                                                              "val_split.json")
if not (os.path.exists(VAL_SPLIT) and os.path.exists(TRAIN_ANN)):
    print("跳过 val-88 复现：未提供冻结 val 划分（SOLAR_VAL_SPLIT）或训练标注 JSON。")
    print("这不影响上面的提交 CSV；可用 make_val_split.py 生成划分后重跑本单元。")
else:
    import solar_eval_local as ev

    val_names = list(json.load(open(VAL_SPLIT, encoding="utf-8"))["val"])
    print(f"冻结 val：{len(val_names)} 图")

    val_prob_dirs = {}
    for s in SEEDS:
        d = os.path.join(WORK, f"prob_val_s{s}_tta")
        os.makedirs(d, exist_ok=True)
        val_prob_dirs[s] = d
        t0, fresh = time.perf_counter(), 0
        for name in val_names:
            stem = os.path.splitext(name)[0]
            dst = os.path.join(d, stem + ".npz")
            if os.path.exists(dst):
                continue
            prob, cls = prob_for_image1024(os.path.join(TRAIN_IMAGES, name),
                                           MODELS[s]["model"], MODELS[s]["dev"],
                                           tta=TTA)
            np.savez_compressed(dst, prob1024=prob.astype(np.float16),
                                cls1024=cls.astype(np.uint8))
            fresh += 1
        print(f"[val s{s}] 新算 {fresh} 张（{time.perf_counter()-t0:.0f}s）", flush=True)

    val_srcs, val_cls, _ = ps.load_prob_dirs([val_prob_dirs[s] for s in ENS_ORDER])
    preds_val, n_inst_val = {}, 0
    for name in val_names:
        stem = os.path.splitext(name)[0]
        pr, cl = ps.upsample2048(ps.avg_sources(val_srcs, None, stem), val_cls[stem])
        lab, n, tag = ps.extract_spec(pr, cl, spec)
        masks = labels_to_masks(lab)
        preds_val[stem] = masks
        n_inst_val += len(masks)

    gt_rows, gt_meta = ev.load_gt_rows(TRAIN_ANN, val_names)
    rep = ev.evaluate(gt_rows, preds_val)
    print(f"GT：{rep['n_rows']} 行 / {gt_meta['gt_images']} 图 / "
          f"{gt_meta['gt_masks']} 根掩膜 | 预测实例 {n_inst_val}")
    print(f"官方池化口径 PQ = {rep['pq']:.6f}"
          f"（当年冻结缓存口径 ≈ 0.344895；TP {rep['tp']} / FP {rep['fp']} / "
          f"FN {rep['fn']}）")
    print(f"Dice（行池化）= {rep['dice_pooled']:.6f}")